<a href="https://colab.research.google.com/github/TriambigaKrubhakaran/_VITECHAbusiveTextDravidianLangtech2026_/blob/main/Google_gemma_2_9B_LoRA_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Tamil Abusive Language Detection using LoRA Fine-tuning
Model: google/gemma-2-9b
Task: Binary Classification (Abusive vs Non-Abusive)
Dataset: 100% Training + 20% Validation (from same set) + Separate Prediction Set

FIXED VERSION - Compatible with all transformers versions
REQUIRES: Google Colab with GPU (T4 or better)
"""

# ============================================================================
# INSTALLATION
# ============================================================================
print("Installing required packages...")
!pip install -q transformers datasets peft accelerate bitsandbytes scikit-learn pandas huggingface_hub
print("✓ Installation complete!\n")

# ============================================================================
# IMPORTS
# ============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
import json
from google.colab import drive
from huggingface_hub import login
import warnings
import gc
import time as time_module
warnings.filterwarnings('ignore')

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# ============================================================================
# HUGGING FACE AUTHENTICATION
# ============================================================================
print("="*70)
print("HUGGING FACE AUTHENTICATION (REQUIRED)")
print("="*70)
print("\nGemma-2 models require authentication.")
print("Get your token from: https://huggingface.co/settings/tokens")
print("Accept license at: https://huggingface.co/google/gemma-2-9b")
print("="*70 + "\n")

hf_token = input("Enter your Hugging Face token: ").strip()
if hf_token:
    try:
        login(token=hf_token)
        print("✓ Successfully authenticated with Hugging Face!")
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        raise
else:
    print("❌ Authentication is required for Gemma-2 models!")
    raise ValueError("HuggingFace token is required")

# ============================================================================
# MOUNT GOOGLE DRIVE
# ============================================================================
print("\n" + "="*70)
print("MOUNTING GOOGLE DRIVE")
print("="*70 + "\n")
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

# ============================================================================
# LOAD TRAINING DATASET (100% FOR TRAINING)
# ============================================================================
print("\n" + "="*70)
print("LOADING TRAINING DATASET (100%)")
print("="*70 + "\n")

train_sheet_url = "https://docs.google.com/spreadsheets/d/1cjL_qfIRvsSy8mbi1vZP61ThjFj2150pCAwBfEsxdcw/export?format=csv&gid=337990985"

train_full_df = pd.read_csv(train_sheet_url)

print(f"✓ Training dataset loaded: {len(train_full_df)} samples")
print(f"✓ Columns: {train_full_df.columns.tolist()}\n")
print("First 5 rows:")
print(train_full_df.head())

# ============================================================================
# LOAD PREDICTION DATASET (SEPARATE)
# ============================================================================
print("\n" + "="*70)
print("LOADING PREDICTION DATASET")
print("="*70 + "\n")

pred_sheet_url = "https://docs.google.com/spreadsheets/d/1r5p4CX9oeQqauyFmkI-nZg3nCDRy4mQqa-XoPm0liRg/export?format=csv&gid=1762241428"

pred_df = pd.read_csv(pred_sheet_url)

print(f"✓ Prediction dataset loaded: {len(pred_df)} samples")
print(f"✓ Columns: {pred_df.columns.tolist()}\n")
print("First 5 rows:")
print(pred_df.head())

# ============================================================================
# PREPROCESS TRAINING DATA
# ============================================================================
print("\n" + "="*70)
print("PREPROCESSING TRAINING DATA")
print("="*70 + "\n")

# Clean column names
train_full_df.columns = train_full_df.columns.str.strip()

# Identify text and label columns
text_col = 'Text'
label_col = 'Class'

if text_col not in train_full_df.columns or label_col not in train_full_df.columns:
    print(f"⚠ Expected columns: 'Text' and 'Class'")
    print(f"⚠ Found columns: {train_full_df.columns.tolist()}")
    text_col = train_full_df.columns[0]
    label_col = train_full_df.columns[1] if len(train_full_df.columns) > 1 else train_full_df.columns[0]

print(f"✓ Text column: '{text_col}'")
print(f"✓ Label column: '{label_col}'")

# Rename columns
train_full_df = train_full_df.rename(columns={text_col: 'text', label_col: 'label'})
train_full_df = train_full_df[['text', 'label']].copy()

# Clean text data
train_full_df['text'] = train_full_df['text'].astype(str).str.strip()
train_full_df = train_full_df[train_full_df['text'] != '']
train_full_df = train_full_df[train_full_df['text'].notna()]

print(f"\nOriginal label distribution:")
print(train_full_df['label'].value_counts())

# Convert labels to binary
train_full_df['label'] = train_full_df['label'].astype(str).str.strip()

label_map = {}
for label in train_full_df['label'].unique():
    label_str = str(label).lower().strip()
    if 'non' in label_str and 'abuse' in label_str:
        label_map[label] = 0
    elif label_str in ['non-abusive', 'non abusive', 'not abusive', 'clean', 'neutral']:
        label_map[label] = 0
    elif 'abuse' in label_str and 'non' not in label_str:
        label_map[label] = 1
    elif label_str in ['abusive', 'offensive', 'toxic', 'hate']:
        label_map[label] = 1
    else:
        label_map[label] = 0

print(f"\nLabel mapping:")
for original, mapped in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  '{original}' → {mapped} ({'Abusive' if mapped == 1 else 'Non-Abusive'})")

train_full_df['label'] = train_full_df['label'].map(label_map).astype(int)
train_full_df = train_full_df.dropna(subset=['text', 'label'])
train_full_df = train_full_df[train_full_df['label'].isin([0, 1])]
train_full_df = train_full_df.reset_index(drop=True)

print(f"\n✓ Cleaned training dataset: {len(train_full_df)} samples")
print(f"\nClass balance:")
print(f"  Non-Abusive (0): {(train_full_df['label'] == 0).sum()} ({(train_full_df['label'] == 0).sum()/len(train_full_df)*100:.2f}%)")
print(f"  Abusive (1): {(train_full_df['label'] == 1).sum()} ({(train_full_df['label'] == 1).sum()/len(train_full_df)*100:.2f}%)")

# Calculate class imbalance ratio
class_counts = train_full_df['label'].value_counts()
imbalance_ratio = class_counts[0] / class_counts[1] if class_counts[1] > 0 else 1.0
print(f"\n⚠️ Class imbalance ratio: {imbalance_ratio:.2f}:1 (Non-Abusive:Abusive)")
if imbalance_ratio > 2.0:
    print(f"  → Will apply class weighting (weight for Abusive class: {imbalance_ratio:.2f})")

# ============================================================================
# DATASET SPLIT: 100% TRAINING + 20% VALIDATION (FROM SAME DATA)
# ============================================================================
print("\n" + "="*70)
print("DATASET SPLIT (100% TRAIN + 20% VALIDATION FROM SAME DATA)")
print("="*70 + "\n")

# Use ALL data for training
train_df = train_full_df.copy()

# Use 20% random sample for validation (overlapping with training)
class_counts = train_full_df['label'].value_counts()
min_class_count = class_counts.min()

if min_class_count < 2:
    _, val_df = train_test_split(train_full_df, test_size=0.2, random_state=42)
else:
    _, val_df = train_test_split(train_full_df, test_size=0.2, random_state=42, stratify=train_full_df['label'])

val_df = val_df.reset_index(drop=True)

print(f"Training: {len(train_df)} samples (100% of dataset)")
print(f"  - Non-Abusive: {(train_df['label'] == 0).sum()}")
print(f"  - Abusive: {(train_df['label'] == 1).sum()}")
print(f"\nValidation: {len(val_df)} samples (20% sample for monitoring)")
print(f"  - Non-Abusive: {(val_df['label'] == 0).sum()}")
print(f"  - Abusive: {(val_df['label'] == 1).sum()}")
print(f"\n⚠️ Note: Validation set overlaps with training (used only for monitoring)")

# ============================================================================
# PREPROCESS PREDICTION DATA
# ============================================================================
print("\n" + "="*70)
print("PREPROCESSING PREDICTION DATA")
print("="*70 + "\n")

# Clean column names
pred_df.columns = pred_df.columns.str.strip()

# Identify text column
pred_text_col = 'Text' if 'Text' in pred_df.columns else pred_df.columns[0]
print(f"✓ Prediction text column: '{pred_text_col}'")

# Prepare prediction dataframe
pred_df = pred_df.rename(columns={pred_text_col: 'text'})
pred_df = pred_df[['text']].copy()

# Clean text data
pred_df['text'] = pred_df['text'].astype(str).str.strip()
pred_df = pred_df[pred_df['text'] != '']
pred_df = pred_df[pred_df['text'].notna()]
pred_df = pred_df.reset_index(drop=True)

print(f"✓ Cleaned prediction dataset: {len(pred_df)} samples")

# ============================================================================
# MODEL AND TOKENIZER LOADING WITH 4-BIT QUANTIZATION
# ============================================================================
print("\n" + "="*70)
print("LOADING MODEL AND TOKENIZER")
print("="*70 + "\n")

model_name = "google/gemma-2-9b"
print(f"Model: {model_name}")
print("Configuration: 4-bit quantization (NF4) for GPU efficiency\n")

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✓ Tokenizer loaded")

# Load model with 4-bit quantization
print("Loading model (this may take 2-3 minutes)...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.id2label = {0: "Non-Abusive", 1: "Abusive"}
model.config.label2id = {"Non-Abusive": 0, "Abusive": 1}

print(f"✓ Model loaded with 4-bit quantization")
print(f"✓ Total parameters: {model.num_parameters():,}")

# Check GPU memory
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"✓ GPU Memory: {allocated:.1f}GB / {mem_gb:.1f}GB")

# ============================================================================
# LORA CONFIGURATION - OPTIMIZED
# ============================================================================
print("\n" + "="*70)
print("CONFIGURING LORA (OPTIMIZED)")
print("="*70 + "\n")

# Determine optimal LoRA rank based on dataset size
dataset_size = len(train_df)
if dataset_size < 5000:
    lora_r = 8
    lora_alpha = 16
    print(f"Dataset size: {dataset_size} → Using r=8 (conservative)")
elif dataset_size < 20000:
    lora_r = 16
    lora_alpha = 32
    print(f"Dataset size: {dataset_size} → Using r=16 (recommended)")
else:
    lora_r = 32
    lora_alpha = 64
    print(f"Dataset size: {dataset_size} → Using r=32 (high capacity)")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    inference_mode=False
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================================
# DATA TOKENIZATION - OPTIMIZED (DYNAMIC PADDING)
# ============================================================================
print("\n" + "="*70)
print("TOKENIZING DATASETS (DYNAMIC PADDING)")
print("="*70 + "\n")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256
    )

# Tokenize training dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Tokenize validation dataset
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("✓ Datasets tokenized with dynamic padding")
print(f"  Training: {len(train_dataset)} samples (100%)")
print(f"  Validation: {len(val_dataset)} samples (20% sample)")
print(f"  ✅ Using DataCollatorWithPadding for efficient batching")

# ============================================================================
# WEIGHTED TRAINER FOR CLASS IMBALANCE
# ============================================================================

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Calculate class weights if imbalanced
if imbalance_ratio > 2.0:
    class_weights = torch.tensor([1.0, imbalance_ratio])
    print(f"\n✓ Using class weights: [1.0, {imbalance_ratio:.2f}]")
else:
    class_weights = None
    print("\n✓ Classes are balanced, no weighting needed")

# ============================================================================
# TRAINING CONFIGURATION - FIXED FOR ALL VERSIONS
# ============================================================================
print("\n" + "="*70)
print("TRAINING CONFIGURATION (FIXED)")
print("="*70 + "\n")

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/gemma2_9b_lora_results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir='/content/drive/MyDrive/gemma2_9b_lora_logs',
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    bf16=True,
    dataloader_num_workers=0,
    seed=42,
    remove_unused_columns=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

print("✓ Training arguments configured (FIXED)")
print(f"  ✅ Learning rate: {training_args.learning_rate}")
print(f"  ✅ LoRA rank: {lora_r}")
print(f"  ✅ Dynamic padding enabled")
print(f"  ✅ Class weighting: {'enabled' if class_weights is not None else 'disabled'}")
print(f"  ✅ evaluation_strategy parameter (compatible with all versions)")

# ============================================================================
# METRICS COMPUTATION
# ============================================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', pos_label=1, zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# ============================================================================
# INITIALIZE TRAINER
# ============================================================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Use weighted trainer if class imbalance detected
if class_weights is not None:
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
else:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

# ============================================================================
# TRAINING TIME ESTIMATION
# ============================================================================
print("\n" + "="*70)
print("TRAINING TIME ESTIMATION")
print("="*70 + "\n")

num_samples = len(train_dataset)
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = num_samples // effective_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Training Configuration:")
print(f"  Total samples: {num_samples}")
print(f"  Effective batch size: {effective_batch_size}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_steps}")
print(f"  Number of epochs: {training_args.num_train_epochs}")

print(f"\n⏱️ Estimated Training Time:")
print(f"  T4 GPU:  {(total_steps * 3.0 / 60):.0f}-{(total_steps * 3.5 / 60):.0f} minutes")
print(f"  L4 GPU:  {(total_steps * 2.0 / 60):.0f}-{(total_steps * 2.3 / 60):.0f} minutes")
print(f"  A100 GPU: {(total_steps * 1.2 / 60):.0f}-{(total_steps * 1.5 / 60):.0f} minutes")

# ============================================================================
# TRAINING
# ============================================================================
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

training_start_time = time_module.time()

try:
    torch.cuda.empty_cache()
    gc.collect()

    trainer.train()

    training_end_time = time_module.time()
    total_training_time = training_end_time - training_start_time

    print("\n" + "="*70)
    print("TRAINING COMPLETED")
    print("="*70)
    print(f"\n⏱️ Total time: {total_training_time/60:.1f} minutes ({total_training_time/3600:.2f} hours)")
    print(f"  Average per step: {total_training_time/total_steps:.2f} seconds")
    print(f"  Training speed: {num_samples/(total_training_time/3600):.0f} samples/hour")
    print("="*70 + "\n")

except Exception as e:
    print(f"\n❌ Training error: {str(e)}")
    import traceback
    traceback.print_exc()

    print("\n⚠ Attempting to save current state...")
    try:
        model.save_pretrained('/content/drive/MyDrive/gemma2_9b_checkpoint')
        print("✓ Checkpoint saved")
    except:
        pass
    raise

# Save model
model_save_path = '/content/drive/MyDrive/gemma2_9b_lora_model'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ Model saved to: {model_save_path}")

# ============================================================================
# VALIDATION SET EVALUATION
# ============================================================================
print("\n" + "="*70)
print("EVALUATING ON VALIDATION SET")
print("="*70 + "\n")

val_predictions = trainer.predict(val_dataset)
val_pred_labels = np.argmax(val_predictions.predictions, axis=1)
val_true_labels = val_df['label'].values

target_names = ['Non-Abusive', 'Abusive']
report = classification_report(
    val_true_labels,
    val_pred_labels,
    target_names=target_names,
    digits=4,
    zero_division=0
)

conf_matrix = confusion_matrix(val_true_labels, val_pred_labels)

print("\n" + "="*70)
print("VALIDATION SET CLASSIFICATION REPORT")
print("="*70 + "\n")
print(report)
print(f"\nConfusion Matrix:")
print(f"{'':20} Predicted")
print(f"{'':20} Non-Abusive  Abusive")
print(f"Actual Non-Abusive   {conf_matrix[0][0]:>11}  {conf_matrix[0][1]:>7}")
print(f"       Abusive       {conf_matrix[1][0]:>11}  {conf_matrix[1][1]:>7}")

# Additional metrics
tn, fp, fn, tp = conf_matrix.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\nAdditional Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")

# ============================================================================
# PREDICTIONS ON SEPARATE DATASET
# ============================================================================
print("\n" + "="*70)
print("GENERATING PREDICTIONS ON SEPARATE DATASET")
print("="*70 + "\n")

# Tokenize prediction data
pred_dataset = Dataset.from_pandas(pred_df[['text']])
pred_dataset = pred_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
pred_dataset.set_format('torch', columns=['input_ids', 'attention_mask'])

print(f"✓ Prediction dataset prepared: {len(pred_dataset)} samples")

# Generate predictions
predictions = trainer.predict(pred_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

print(f"✓ Predictions generated for {len(pred_labels)} samples")

# ============================================================================
# SAVE PREDICTIONS
# ============================================================================
print("\n" + "="*70)
print("SAVING PREDICTIONS")
print("="*70 + "\n")

# Create predictions dataframe with predicted labels
predictions_df = pd.DataFrame({
    'predicted_label': pred_labels
})

# Save to CSV
predictions_path = '/content/drive/MyDrive/gemma2_9b_predictions.csv'
predictions_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')
print(f"✓ Predictions saved: {predictions_path}")

# Show prediction distribution
print(f"\nPrediction Distribution:")
print(f"  Non-Abusive (0): {(pred_labels == 0).sum()} ({(pred_labels == 0).sum()/len(pred_labels)*100:.2f}%)")
print(f"  Abusive (1): {(pred_labels == 1).sum()} ({(pred_labels == 1).sum()/len(pred_labels)*100:.2f}%)")

# Show first 10 predictions
print(f"\nFirst 10 Predictions:")
for i in range(min(10, len(predictions_df))):
    label_name = 'Abusive' if predictions_df.iloc[i]['predicted_label'] == 1 else 'Non-Abusive'
    print(f"  {i+1}. {label_name} ({predictions_df.iloc[i]['predicted_label']})")

# ============================================================================
# SAVE COMPREHENSIVE REPORT
# ============================================================================
print("\n" + "="*70)
print("SAVING COMPREHENSIVE REPORT")
print("="*70 + "\n")

report_path = '/content/drive/MyDrive/gemma2_9b_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("TAMIL ABUSIVE LANGUAGE DETECTION - GEMMA-2-9B\n")
    f.write("="*80 + "\n\n")
    f.write(f"Model: {model_name}\n")
    f.write(f"Configuration: 100% Training + 20% Validation + Separate Prediction\n\n")
    f.write(f"Dataset:\n")
    f.write(f"  - Training samples: {len(train_df)} (100%)\n")
    f.write(f"  - Validation samples: {len(val_df)} (20% sample)\n")
    f.write(f"  - Prediction samples: {len(pred_df)}\n")
    f.write(f"  - Class imbalance ratio: {imbalance_ratio:.2f}:1\n\n")
    f.write(f"LoRA Configuration:\n")
    f.write(f"  - Rank (r): {lora_config.r}\n")
    f.write(f"  - Alpha: {lora_config.lora_alpha}\n")
    f.write(f"  - Dropout: {lora_config.lora_dropout}\n\n")
    f.write(f"Training Time:\n")
    f.write(f"  - Total: {total_training_time/60:.1f} minutes\n")
    f.write(f"  - Per step: {total_training_time/total_steps:.2f} seconds\n\n")
    f.write("="*80 + "\n")
    f.write("VALIDATION SET CLASSIFICATION REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(report)
    f.write(f"\n\nConfusion Matrix:\n")
    f.write(f"{'':20} Predicted\n")
    f.write(f"{'':20} Non-Abusive  Abusive\n")
    f.write(f"Actual Non-Abusive   {conf_matrix[0][0]:>11}  {conf_matrix[0][1]:>7}\n")
    f.write(f"       Abusive       {conf_matrix[1][0]:>11}  {conf_matrix[1][1]:>7}\n")
    f.write(f"\n\nPrediction Dataset Distribution:\n")
    f.write(f"  Non-Abusive (0): {(pred_labels == 0).sum()} ({(pred_labels == 0).sum()/len(pred_labels)*100:.2f}%)\n")
    f.write(f"  Abusive (1): {(pred_labels == 1).sum()} ({(pred_labels == 1).sum()/len(pred_labels)*100:.2f}%)\n")

print(f"✓ Report saved: {report_path}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✅ TRAINING AND PREDICTION COMPLETE")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Training samples: {len(train_df)} (100%)")
print(f"  Validation samples: {len(val_df)} (20% sample)")
print(f"  Prediction samples: {len(pred_df)}")
print(f"  Training time: {total_training_time/60:.1f} minutes")

acc = accuracy_score(val_true_labels, val_pred_labels)
prec, rec, f1, _ = precision_recall_fscore_support(val_true_labels, val_pred_labels, average='binary', pos_label=1)
print(f"\n📈 Validation Performance:")
print(f"  Accuracy: {acc:.4f}")
print(f"  Precision (Abusive): {prec:.4f}")
print(f"  Recall (Abusive): {rec:.4f}")
print(f"  F1-Score (Abusive): {f1:.4f}")
print(f"  Sensitivity: {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")

print(f"\n📁 Output Files:")
print(f"  Model: {model_save_path}")
print(f"  Predictions: {predictions_path}")
print(f"  Report: {report_path}")

print(f"\n✅ Configuration:")
print(f"  ✅ 100% of training data used for training")
print(f"  ✅ 20% sample used for validation monitoring")
print(f"  ✅ Separate dataset predictions saved")
print(f"  ✅ Output contains only predicted_label column (0 or 1)")
print("="*80)